## Bert (Encoder-Only)

Bert stands for: Bidirectional Encoder Representations from Transformers. Bert is based on the encoder-only block of the transformer. Bert Does not generate text from left-to-right the way GPT2 and other decoder-only based models do. Bert instead performs Masked-Language Modeling(MLM). Instead gpt based models where they predict the next word based on all words that came before it, Bert masks the word in the middle of the sequence and asks the model to predict it. Bert is good for text-classification tasks. Bert does not generate words as output. The reason why it is important to mask the words, is so context is contained in the sequence and bert understands it.

### Example of Context Using lyrics from Gucci Mane's song "Lemonade"
**Warning, offensive content**

- **Phrase 1** My phantom sit on sixes no 20s in my denim. your cutlass knocking because it is a `lemon`.
- **Phrase 2** I like them Georgia Peaches but you look more like a `lemon`.
- **Phrase 3** I'm pimping wearing linen, that's just how I am chillin. I'm smoking grits and selling chickens Corvette painted `lemon`.
- **Phrase 4** I got lemonade and `lemon` tint.
- **Phrase 5** Half a pound of `lemon` kush, call that pack the "`lemon` drop".
- **Phrase 6** Just stash one `lemon`, homie, I can supply damn near 20 blocks.

Meaning, from **phrase 1**, when Bert (or another MLM-based model) masks the word 
`lemon`, it does not only look at the words to the left like GPT would. It attends 
to the tokens on *both* sides of the mask at the same time. The words "cutlass" 
and "knocking" before the mask, and the sentence structure around it, tell Bert 
that the masked word probably describes a faulty car — so `lemon` here means a 
broken-down vehicle, not a fruit.

In **phrase 2**, the surrounding tokens "Georgia Peaches" and "you look more like" 
push the meaning toward an insult about someone's looks. In **phrase 5**, the 
neighbors "kush" and "pack" push it toward a cannabis strain. Same exact token, 
completely different learned representation, purely because of the context on 
both sides.

This is the core idea behind the "B" in Bert: **bidirectional**. Because the 
model sees the full sequence (minus the masked tokens) during training, the 
embedding it produces for `lemon` is different in every phrase. This is why Bert 
is strong at tasks like text classification, named entity recognition, and 
sentence similarity — tasks that need deep *understanding* of a sequence rather 
than *generation* of new text.

In [1]:
import torch
from transformers import AutoModel, AutoTokenizer

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()

phrases = [
    "My phantom sit on sixes no 20s in my denim. Your cutlass knocking because it is a lemon.",
    "I like them Georgia peaches but you look more like a lemon.",
    "I'm pimping wearing linen, that's just how I am chilling. I'm smoking grits and selling chickens. Corvette painted lemon.",
    "I got lemonade and lemon tint.",
    "Half a pound of lemon kush, call that pack the lemon drop.",
    "Just stash one lemon, homie, I can supply damn near 20 blocks.",
]

inputs = tokenizer(phrases, padding=True, return_tensors="pt")

with torch.inference_mode():
    hidden_states = model(**inputs).last_hidden_state

lemon_token_id = tokenizer.convert_tokens_to_ids("lemon")
lemon_embeddings = []

for phrase_index, token_ids in enumerate(inputs["input_ids"]):
    lemon_positions = (token_ids == lemon_token_id).nonzero(as_tuple=True)[0]

    for lemon_position in lemon_positions:
        embedding = hidden_states[phrase_index, lemon_position]
        lemon_embeddings.append(embedding)

        print(f"Phrase {phrase_index + 1}")
        print("Token ID:", token_ids[lemon_position].item())
        print("Embedding preview:", embedding[:8].tolist())
        print()

embeddings = torch.stack(lemon_embeddings)

/home/nick/github-projects/bert-t5-gpt2/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4954.01it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be igno

Phrase 1
Token ID: 14380
Embedding preview: [-0.3763357400894165, 0.14407895505428314, -0.08382433652877808, -0.170987069606781, -0.12634599208831787, 0.6208314895629883, 0.06525921076536179, 0.743113100528717]

Phrase 2
Token ID: 14380
Embedding preview: [-0.20102892816066742, 0.1790442168712616, -0.45599856972694397, 0.33063629269599915, -0.09839607030153275, 1.0829777717590332, 0.1509000062942505, 1.203596591949463]

Phrase 3
Token ID: 14380
Embedding preview: [0.42821717262268066, 0.2820999324321747, -0.31471380591392517, 0.153246209025383, 0.7198457717895508, 0.40720275044441223, -0.3256567120552063, 0.959901750087738]

Phrase 4
Token ID: 14380
Embedding preview: [0.04446505382657051, 0.5790777802467346, -0.19395478069782257, 0.28629279136657715, 0.05566851794719696, 0.5046402812004089, -0.18547016382217407, 0.7593722343444824]

Phrase 4
Token ID: 14380
Embedding preview: [0.06277823448181152, 0.531137228012085, -0.2742941677570343, 0.6189624071121216, -0.10846485942602158, 0.0761

### Explanation of output

The output cell is displaying four things:
1. sends each lyric through bert separately
2. finds each standalone `lemon` token. Every single `lemon` token has the same token ID.
3. Extracts BERT’s final 768-number embedding for that specific occurrence.
4. Collects those embeddings into `lemon embeddings` so the next cell can compare or classify them.

The main difference from a decoder-only autoregressive model is what information the model can use for each word.

BERT creates the embedding for `lemon` after reading the words on both its left and its right. In this example, `cutlass knocking` before `lemon` and the following words in each phrase can both affect BERT's final 768-number representation.

A decoder-only model such as GPT-2 processes text left to right. At the position of `lemon`, it can only use the words that appeared before `lemon`; it cannot use words that come afterward when representing that position. Its main task is to predict the next token, not to create a bidirectional representation of a masked word.

Therefore, BERT is useful when the full context is available and the goal is understanding or classifying text. Decoder-only models are useful when the goal is generating the next tokens in a sequence.

### Prototype Classification

This cell uses BERT's contextual embeddings to assign a label to each occurrence of `lemon`.

First, it selects one previously labeled embedding as a prototype for each category: `broken car`, `insult`, `color`, `marijuana`, and `selling`. It then compares every `lemon` embedding to those prototypes using cosine similarity.

The predicted label is the prototype with the highest similarity score. A score of `1.000` means the occurrence is being compared with itself, because the prototype examples are taken from the same small set of phrases.

This is a demonstration of how contextual BERT embeddings can support classification. It is not a trained classifier, so the labels come from the examples selected by hand and the score is similarity, not a calibrated probability or true confidence.

In [2]:
prototype_indices = {
    "broken car": 0,
    "insult": 1,
    "color": 2,
    "marijuana": 4,
    "selling": 6,
}

prototypes = torch.stack(
    [embeddings[index] for index in prototype_indices.values()]
)
class_names = list(prototype_indices)

scores = torch.nn.functional.cosine_similarity(
    embeddings.unsqueeze(1),
    prototypes.unsqueeze(0),
    dim=2,
)

for occurrence_index, score_row in enumerate(scores):
    predicted_index = score_row.argmax().item()
    predicted_class = class_names[predicted_index]
    confidence = score_row[predicted_index].item()

    print(
        f"Occurrence {occurrence_index + 1}: "
        f"{predicted_class} ({confidence:.3f})"
    )

Occurrence 1: broken car (1.000)
Occurrence 2: insult (1.000)
Occurrence 3: color (1.000)
Occurrence 4: marijuana (0.865)
Occurrence 5: marijuana (1.000)
Occurrence 6: selling (0.833)
Occurrence 7: selling (1.000)
Occurrence 8: broken car (0.752)


### Interpreting the Output

Each output line gives one occurrence of `lemon`, its predicted label, and its cosine-similarity score to the closest prototype.

For example, `Occurrence 1: broken car (1.000)` means the first `lemon` embedding was closest to the hand-labeled `broken car` prototype. Its score is exactly `1.000` because occurrence 1 itself was chosen as that prototype.

The other scores show which prototype BERT considers most similar based on the full phrase context. A larger score means the contextual embeddings are closer; it does not mean BERT is certain or that the classification is guaranteed correct.

Because the prototype examples came from this same small lyric set, this is an illustration of contextual embedding similarity rather than a trained semantic classifier. A real classifier would need many independently labeled lyric examples for each meaning.

**Note**: BERT produces a contextual embedding for every token in every input sequence. This example extracts only the embeddings for `lemon` so their different contextual meanings can be compared.